# SIT223/SIT753 7.3HD - Jenkins DevOps Pipeline

**Executed implementation and verification notebook**

This notebook records the project requirements, implementation steps, and reproducible checks for the seven-stage pipeline. It does not print secrets, and Jenkins remains responsible for deployment.


## 1. Problem understanding and Appendix I validation

The assessment requires an own project, a Git repository, and seven functioning Jenkins stages: Build, Test, Code Quality, Security, Deploy, Release, and Monitoring. The top band additionally expects smooth automation, artefact retention, deployment best practices, rollback support, immutable/versioned releases, live production monitoring, meaningful alerts, and an incident simulation.

Stockroom is suitable under Appendix I because it is not static content: it combines a browser interface with authenticated Flask CRUD endpoints, validation, SQLite persistence, health/version endpoints, and Prometheus metrics. It has a repeatable wheel and Docker build, 16 unit/integration tests, static quality and security tooling, container deployments, a production promotion path, rollback automation, and observable failure/recovery behaviour.


## 2. Execution plan

1. Keep application scope compact but functionally complete: UI, backend logic, persistence, authentication, validation, health, version, and metrics.
2. Gate a wheel and Docker image with automated tests, branch coverage, lint/format rules, and security scans.
3. Deploy the same tested image to staging, smoke-test it, then promote an immutable semantic version to production.
4. Prove rollback by temporarily deploying the previous retained image and restoring the current release.
5. Monitor production with Prometheus and Alertmanager, then automate an incident and recovery check.
6. Archive machine-readable evidence and document the exact Jenkins/GitHub setup for the demonstration.


## 3. Project identity and reproducibility


In [1]:
import json
import re
import subprocess
import sys
import tempfile
from pathlib import Path

root = Path.cwd().resolve()
assert (root / "Jenkinsfile").exists(), "Run this notebook from the repository root"
project_python = root / ".venv/bin/python"
if not project_python.exists():
    project_python = Path(sys.executable)

version = (root / "VERSION").read_text().strip()
runtime = ".venv/bin/python" if project_python == root / ".venv/bin/python" else "current Python kernel"
print(f"Repository: {root.name}")
print(f"Validation runtime: {runtime}")
print(f"Release version: {version}")


Repository: 7.3HD_Inventory_DevOps
Validation runtime: .venv/bin/python
Release version: 1.1.1


## 4. Application-level functional check


In [2]:
functional_check = r'''from pathlib import Path
import tempfile
from inventory_api import create_app

with tempfile.TemporaryDirectory() as temporary_directory:
    app = create_app({
        "TESTING": True,
        "DATABASE": str(Path(temporary_directory) / "inventory.db"),
        "API_KEY": "not-a-production-secret",
        "ENVIRONMENT": "notebook",
    })
    client = app.test_client()
    headers = {"X-API-Key": "not-a-production-secret"}
    assert client.get("/").status_code == 200
    assert client.get("/health").get_json() == {"database": True, "status": "ok"}
    assert client.get("/api/items").status_code == 401
    created = client.post(
        "/api/items",
        json={"sku": "NB-001", "name": "Notebook evidence item", "quantity": 3},
        headers=headers,
    )
    assert created.status_code == 201
    item_id = created.get_json()["id"]
    updated = client.put(
        f"/api/items/{item_id}",
        json={"sku": "NB-001", "name": "Notebook evidence item", "quantity": 5},
        headers=headers,
    )
    assert updated.get_json()["quantity"] == 5
    assert client.delete(f"/api/items/{item_id}", headers=headers).status_code == 200
    assert "inventory_readiness" in client.get("/metrics").get_data(as_text=True)
print("PASS: UI, readiness, authentication, SQLite CRUD, and metrics verified in-process.")'''
functional = subprocess.run(
    [str(project_python), "-c", functional_check],
    cwd=root,
    text=True,
    capture_output=True,
)
print(functional.stdout)
if functional.stderr:
    print(functional.stderr)
assert functional.returncode == 0


PASS: UI, readiness, authentication, SQLite CRUD, and metrics verified in-process.



## 5. Test and branch-coverage gate


In [3]:
test = subprocess.run(
    [
        str(project_python), "-m", "pytest", "-q",
        "--cov=inventory_api", "--cov-branch",
        "--cov-report=term-missing", "--cov-fail-under=90",
    ],
    cwd=root,
    text=True,
    capture_output=True,
)
print(test.stdout)
if test.stderr:
    print(test.stderr)
assert test.returncode == 0


................                                                         [100%]
================================ tests coverage ================================
______________ coverage: platform darwin, python 3.12.14-final-0 _______________

Name                              Stmts   Miss Branch BrPart  Cover   Missing
-----------------------------------------------------------------------------
src/inventory_api/__init__.py         2      0      0      0   100%
src/inventory_api/app.py            105      5     16      4    93%   29, 36->39, 133, 140, 143-144
src/inventory_api/db.py              49      2      2      0    96%   104-105
src/inventory_api/validation.py      22      0     14      0   100%
-----------------------------------------------------------------------------
TOTAL                               178      7     32      4    95%
Required test coverage of 90% reached. Total coverage: 94.76%
16 passed in 0.09s



## 6. Code-quality gates


In [4]:
quality_paths = ["src", "tests", "scripts", "monitoring"]
quality_commands = [
    [str(project_python), "-m", "ruff", "check", *quality_paths, "--output-format=concise"],
    [str(project_python), "-m", "ruff", "format", "--check", *quality_paths],
]
for command in quality_commands:
    result = subprocess.run(command, cwd=root, text=True, capture_output=True)
    print("$", " ".join(command[2:]))
    print((result.stdout + result.stderr).strip() or "PASS")
    assert result.returncode == 0


$ ruff check src tests scripts monitoring --output-format=concise
All checks passed!
$ ruff format --check src tests scripts monitoring
9 files already formatted


## 7. Security controls


In [5]:
with tempfile.TemporaryDirectory() as temporary_directory:
    bandit_report = Path(temporary_directory) / "bandit.json"
    bandit = subprocess.run(
        [str(project_python), "-m", "bandit", "-q", "-r", "src", "scripts", "monitoring", "-ll", "-f", "json", "-o", str(bandit_report)],
        cwd=root,
        text=True,
        capture_output=True,
    )
    report = json.loads(bandit_report.read_text())
    print(f"Bandit blocking findings: {len(report.get('results', []))}")
    assert bandit.returncode == 0 and not report.get("results")

jenkinsfile = (root / "Jenkinsfile").read_text()
dockerfile = (root / "Dockerfile").read_text()
production_compose = (root / "deploy/docker-compose.production.yml").read_text()
assert "withCredentials" in jenkinsfile and "inventory-api-key" in jenkinsfile
assert "USER app" in dockerfile
assert "INVENTORY_API_KEY:?INVENTORY_API_KEY is required" in production_compose
print("PASS: masked Jenkins credential, mandatory production secret, and non-root runtime verified.")
print("pip-audit is also a blocking Jenkins gate; its JSON report is archived by every pipeline run.")


Bandit blocking findings: 0
PASS: masked Jenkins credential, mandatory production secret, and non-root runtime verified.
pip-audit is also a blocking Jenkins gate; its JSON report is archived by every pipeline run.


## 8. Seven-stage pipeline structure


In [6]:
jenkinsfile = (root / "Jenkinsfile").read_text()
observed = re.findall(r"stage\('([^']+)'\)", jenkinsfile)
assessed = ["Build", "Test", "Code Quality", "Security", "Deploy", "Release", "Monitoring"]
assert all(stage in observed for stage in assessed)
print("Observed Jenkins nodes:", " → ".join(observed))
print("Assessed stages (7):", ", ".join(assessed))

checks = {
    "staging target": "STAGING_URL = 'http://127.0.0.1:8081'",
    "production target": "PRODUCTION_URL = 'http://127.0.0.1:8082'",
    "rollback parameter": "RUN_ROLLBACK_TEST",
    "incident parameter": "RUN_INCIDENT_TEST",
    "production monitor job": "job_name: inventory-api-production",
    "critical down alert": "alert: InventoryApiDown",
}
sources = jenkinsfile + (root / "monitoring/prometheus.yml").read_text() + (root / "monitoring/alerts.yml").read_text()
for name, token in checks.items():
    assert token in sources, name
    print(f"PASS: {name}")


Observed Jenkins nodes: Checkout → Build → Test → Code Quality → Security → Deploy → Release → Monitoring
Assessed stages (7): Build, Test, Code Quality, Security, Deploy, Release, Monitoring
PASS: staging target
PASS: production target
PASS: rollback parameter
PASS: incident parameter
PASS: production monitor job
PASS: critical down alert


## 9. Final Jenkins execution evidence

![Successful Jenkins build 6 showing all stages](evidence/jenkins-pipeline-build-6-viewport.png)

Jenkins build **#6** completed successfully in **2 min 19 sec** from GitHub commit `d32a22b` and released version **1.1.1**. It ran **16 tests**, reached **94.76% branch coverage**, passed Ruff, Bandit, and pip-audit, completed authenticated staging and production smoke tests, demonstrated rollback to **1.1.0** and restoration to **1.1.1**, and verified the production incident sequence from alert firing through Alertmanager delivery, service recovery, and resolved notification.

Archived evidence includes the wheel, dependency resolution, image metadata, JUnit and coverage XML, Bandit and pip-audit JSON, release manifest, Compose logs, and `rollback-evidence.txt`.


## 10. Result and reflection

All seven assessed stages run as blocking Jenkins gates. The Release stage promotes the Docker image that passed testing and adds the semantic version tag; it does not rebuild the application for production. The rollback and incident checks also run the recovery steps and archive their results.

The pipeline runs on one local Jenkins node. Staging and production use separate Compose configurations, Jenkins injects the production API key, and health checks stop a failed deployment. Prometheus reads production metrics, while the incident test confirms that an alert is delivered and resolved after the service recovers.
